# Sample Builder Outputs EDA

This notebook explores how SampleBuilder configurations (provided through datamodules) behave on the TACO v1.4 trace datasets (a selected set of shards, or all shards). We measure what kind of samples are planned for generation, and what kind of samples are actually generated at runtime (given trace data constraints).

In [ ]:
import ast
import collections
import contextlib
import json
import random
import typing

import IPython.display as ipy_display
import matplotlib.pyplot as plt
import pandas as pd
import tqdm

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.organisms.datamodules.samples as sample_utils
import pyine.organisms.datamodules.shortcuts as shortcuts
import pyine.organisms.datamodules.shortcuts_configs as shortcuts_configs
import pyine.utils.pydantic

plt.style.use("ggplot")

In [ ]:
# ------------ CHANGE THESE SETTINGS IF NEEDED ------------
source_dataset_name = "TACO"  # by default, we target the TACO 10s10t traces dataset
trace_dataset_pattern = "v1.4/10s10t.*of000026.*.lmdb"  # specific pattern for the v1.4 dataset
expected_part_count = 26  # given the 500-problem-chunk split used for the v1.4 dataset
selected_part_indices: int | list[int] = list(range(13))  # list of zero-based indices
target_datamodule = "shortcuts"
target_subsets = ["train", "valid"]
max_samples_per_subset = None  # None = no maximum
use_hybrid_sample_transforms = True
seed = 0
epoch = 0
# ---------------------------------------------------------

In [ ]:
# prepare the target datamodule with its default config given the provided dataset info
dataset_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name=source_dataset_name,
        pattern=trace_dataset_pattern,
    )
)
if isinstance(selected_part_indices, int):
    selected_part_indices = [selected_part_indices]
assert isinstance(selected_part_indices, list) and len(selected_part_indices) > 0, "missing shard selection"
assert all(0 <= idx < len(dataset_paths) for idx in selected_part_indices), "invalid shard selection"
dataset_paths = [dataset_paths[idx] for idx in selected_part_indices]
print("will perform analysis on the following dataset shards:")
for path in dataset_paths:
    print(f"- {path}")
supported_datamodules = ["shortcuts"]
if target_datamodule not in supported_datamodules:
    raise ValueError(f"unsupported datamodule: {target_datamodule}; pick one of {supported_datamodules}")
elif target_datamodule == "shortcuts":
    default_dm_config = shortcuts_configs.get_datamodule_config(
        lmdb_paths=dataset_paths,
        split_file_path=pyine.data.utils.splits.get_dataset_split_file_path(source_dataset_name),
        seed=seed,
        use_hybrid_sample_transforms=use_hybrid_sample_transforms,
        as_pydantic=True,
    )
    datamodule = shortcuts.ShortcutBiasDataModule(default_dm_config, verbose=True)
else:
    raise NotImplementedError(f"missing impl for datamodule: {target_datamodule}")
print("performing datamodule preparation + setup...")
datamodule.prepare_data()
datamodule.setup()
for subset_name in target_subsets:
    parser = datamodule.get_parser(subset_name)
    parser.set_epoch(epoch)
    print(f"- {subset_name} parser will generate {len(parser)} samples per epoch")
print("datamodule ready-to-go!")

In [ ]:
def collect_sample_rows(
    builder: sample_utils.SampleBuilder,
    subset_name: str,
    max_samples: int | None = None,
) -> list[dict[str, typing.Any]]:
    """Collect tabular rows describing generated samples.

    Args:
        builder: SampleBuilder to draw samples from.
        subset_name: Subset name associated with the builder.
        max_samples: Optional cap on the number of samples to collect.

    Returns:
        List of dictionaries containing summary information for each sampled entry.
    """
    sample_rows: list[dict[str, typing.Any]] = []
    sample_cap = len(builder) if max_samples is None else min(len(builder), max_samples)
    sample_idx_iter = tqdm.tqdm(range(sample_cap), desc="parsing samples", smoothing=0.1)
    for sample_idx in sample_idx_iter:
        sample = builder[sample_idx]
        tag_list = [tag for tag in sample.comma_separated_tags.split(",") if tag]
        sample_rows.append(
            {
                "subset": subset_name,
                "identifier": sample.identifier,
                "code_line_count": len(sample.code.splitlines()),
                "code_target_line_span": sample.last_line - sample.first_line,
                "description_word_count": len(sample.description.split()),
                "inputs_char_length": len(sample.inputs),
                "expected_output_char_length": len(sample.expected_output),
                "predict_type": sample.predict_type,
                "code_type": sample.code_type,
                "trace_step_count": sample.trace_step_count,
                "has_code_override": sample.has_code_override,
                "tags": tag_list,
            }
        )
    return sample_rows


builder_map: dict[str, sample_utils.SampleBuilder] = {}
sample_rows: list[dict[str, typing.Any]] = []
for subset_name in target_subsets:
    parser = datamodule.get_parser(subset_name)
    assert isinstance(parser, sample_utils.SampleBuilder), f"unexpected parser type: {type(parser)}"
    if not len(parser):
        print(f"Skipping {subset_name}: no samples found.")
        continue
    builder_map[subset_name] = parser
    print(f"collecting samples for {subset_name} subset...")
    rows = collect_sample_rows(
        builder=parser,
        subset_name=subset_name,
        max_samples=max_samples_per_subset,
    )
    sample_rows.extend(rows)
print(f"sample collection complete (total samples: {len(sample_rows)})")

In [ ]:
if sample_rows:
    samples_df = pd.DataFrame(sample_rows)
    ipy_display.display(samples_df.head())
else:
    samples_df = pd.DataFrame()
    print("no samples collected; ensure the dataset is available and contains traces")

In [ ]:
if builder_map:
    stats_rows = []
    expected_probs_map = {}
    for subset_name, builder in builder_map.items():
        stats = builder.get_stats()
        stats_rows.append({"subset": subset_name, **stats})
        expected_probs_map[subset_name] = builder.selection_config.code_type_prob_map
    stats_df = pd.DataFrame(stats_rows).set_index("subset").fillna(0)
    ipy_display.display(stats_df)
    print(f"{stats_df.columns=}")

    # also create a df for ratio'd statistics
    prefixes_to_skip = ["orig_trace", "kept_trace", "sample_count", "summaries_count", "filtered/"]
    prefixes_to_ratio = ["selected/", "code_overrides_count"]
    ratios_df_cols = [c for c in stats_df.columns if not any(c.startswith(s) for s in prefixes_to_skip)]
    ratios_df_target_cols = [c for c in ratios_df_cols if any(c.startswith(s) for s in prefixes_to_ratio)]
    ratios_df = pd.DataFrame(stats_df[ratios_df_cols], index=stats_df.index)
    ratios_df[ratios_df_target_cols] = ratios_df[ratios_df_target_cols].div(stats_df["sample_count"].squeeze(), axis=0)
    ratios_df = ratios_df.rename(columns=lambda c: c.replace("count", "ratio"))
    expected_prob_rows = [
        {f"selected/type/{input_type}": prob for input_type, prob in expected_probs_map[subset_name].items()}
        for subset_name in target_subsets
    ]
    ratios_df = pd.concat(
        [
            ratios_df,
            pd.DataFrame(expected_prob_rows, index=[f"{s}_expected" for s in target_subsets]),
        ]
    )
    ratios_df = ratios_df.sort_index()
    ipy_display.display(ratios_df)
else:
    print("no builds created; statistics are unavailable")

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    metric_definitions = [
        ("code_line_count", "Code length (lines)", {"bins": 30}),
        ("code_target_line_span", "Target code line span", {"bins": 30}),
        ("description_word_count", "Description length (words)", {"bins": 30}),
        ("inputs_char_length", "Input args length (chars)", {"bins": 50}),
        ("expected_output_char_length", "Expected output length (chars)", {"bins": 50}),
        ("trace_step_count", "Trace step count", {"bins": 50}),
    ]
    for metric_column, metric_title, plot_kwargs in metric_definitions:
        fig, axes = plt.subplots(
            1,
            len(target_subsets),
            figsize=(6 * len(target_subsets), 4),
            sharey=True,
        )
        if len(target_subsets) == 1:
            axes = [axes]
        for axis, subset_name in zip(axes, target_subsets, strict=False):
            subset_df = samples_df[samples_df["subset"] == subset_name]
            if subset_df.empty:
                axis.text(0.5, 0.5, "No samples", ha="center", va="center")
                axis.set_title(f"[{subset_name}]")
                axis.set_xlabel(metric_title)
                axis.set_ylabel("Sample count")
                continue
            axis.hist(subset_df[metric_column], color="#2a9d8f", alpha=0.85, **plot_kwargs)
            axis.set_title(f"[{subset_name}]")
            axis.set_xlabel(metric_title)
            axis.set_ylabel("Sample count")
            axis.set_yscale("log")
        if len(target_subsets) > 1:
            fig.suptitle(metric_title)
        plt.tight_layout()
        plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # create a shared color mapping for predict types
    predict_types = sorted(samples_df["predict_type"].unique())
    color_palette = plt.cm.tab10.colors
    predict_type_colors = {pt: color_palette[idx % len(color_palette)] for idx, pt in enumerate(predict_types)}

    # left plot: predict type distribution by subset (count)
    predict_type_counts = samples_df.groupby(["subset", "predict_type"]).size().unstack(fill_value=0).sort_index(axis=1)
    bar_colors = [predict_type_colors[pt] for pt in predict_type_counts.columns]
    predict_type_counts.plot(kind="bar", ax=axes[0], color=bar_colors)
    axes[0].set_title("Predict type distribution by subset")
    axes[0].set_ylabel("Sample count")
    axes[0].tick_params(axis="x", rotation=0)
    ymax = predict_type_counts.values.max()
    axes[0].set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in axes[0].containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        axes[0].bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    # right plot: trace step count distribution by predict type (boxplot)
    boxplot_data = [samples_df[samples_df["predict_type"] == pt]["trace_step_count"].values for pt in predict_types]
    bp = axes[1].boxplot(boxplot_data, tick_labels=predict_types, patch_artist=True)
    for patch, pt in zip(bp["boxes"], predict_types, strict=False):
        patch.set_facecolor(predict_type_colors[pt])
        patch.set_alpha(0.7)
    axes[1].set_title("Trace step count by predict type")
    axes[1].set_ylabel("Trace step count")
    axes[1].set_xlabel("Predict type")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_yscale("log")

    plt.tight_layout()
    plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    code_type_counts = samples_df.groupby(["subset", "code_type"]).size().unstack(fill_value=0).sort_index(axis=1)
    ax = code_type_counts.plot(kind="bar", figsize=(10, 6))
    plt.title("Code type distribution by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = code_type_counts.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    tag_counter = collections.Counter()
    for tag_list in samples_df["tags"]:
        tag_counter.update(tag_list)
    most_common_tags = tag_counter.most_common(20)
    if not most_common_tags:
        print("no tags identified in the sampled data")
    else:
        tag_labels, tag_values = zip(*most_common_tags, strict=False)
        total_samples = len(samples_df)
        proportions = [v / total_samples for v in tag_values]

        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.bar(tag_labels, tag_values, color="#264653", alpha=0.9)
        ax.set_ylabel("Occurrences")
        ax.set_title("Top 20 tags across sampled data")
        ax.set_xticks(range(len(tag_labels)))
        ax.set_xticklabels(tag_labels, rotation=45, ha="right")

        percent_labels = [f"{p * 100:.1f}%" for p in proportions]
        ax.bar_label(bars, labels=percent_labels, padding=3, fontsize=9, color="#1d3557")

        ymax = max(tag_values) if tag_values else 1
        ax.set_ylim(0, ymax * 1.15)
        ax.grid(axis="y", linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    override_summary = (
        samples_df.groupby(["subset", "has_code_override"])
        .size()
        .unstack(fill_value=0)
        .rename(
            columns={
                False: "no override",
                True: "with override",
            }
        )
    )
    ax = override_summary.plot(kind="bar", figsize=(8, 5))
    plt.title("Code override counts by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = override_summary.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()

In [ ]:
# ------------ SAMPLE VIEWER SETTINGS ------------
viewer_target_subset: str = "train"  # which subset to pick from
viewer_sample_index: int | None = None  # specific index (0-based), or None for random
viewer_max_string_length: int = 2000  # truncate long strings beyond this length (0 = no limit)
sample_until_code_type: str | None = None  # keep randomly sampling until hitting this code type
sample_until_pred_type: str | None = None  # keep randomly sampling until hitting this pred type
# -------------------------------------------------


def _format_long_string(value: str, max_len: int) -> str:
    """Format a potentially long string with optional truncation."""
    if 0 < max_len < len(value):
        truncated = value[:max_len]
        return f"{truncated}\n\n... [truncated, showing {max_len}/{len(value)} chars]"
    return value


def _format_dict_string(value: str, max_len: int) -> tuple[str, bool]:
    """Try to parse and pretty-print JSON or Python dict literal; returns (formatted_str, success)."""
    parsed = None
    # try JSON first
    with contextlib.suppress(json.JSONDecodeError, TypeError):
        parsed = json.loads(value)
    # try Python literal (handles single-quoted dicts)
    if parsed is None:
        with contextlib.suppress(SyntaxError, ValueError):
            parsed = ast.literal_eval(value)
    if parsed is not None and isinstance(parsed, (dict, list)):
        formatted = json.dumps(parsed, indent=2)
        return _format_long_string(formatted, max_len), True
    return _format_long_string(value, max_len), False


def display_sample(
    sample: sample_utils.SampleData,
    max_string_length: int = 2000,
) -> None:
    """Display a sample's full content with nice formatting."""
    # metadata section (without tags/metrics, which are displayed separately)
    metadata_md = f"""
## Sample Metadata

| Field | Value |
|-------|-------|
| **Identifier** | `{sample.identifier}` |
| **Predict Type** | `{sample.predict_type}` |
| **Code Type** | `{sample.code_type}` |
| **Entrypoint** | `{sample.entrypoint or "(none)"}` |
| **Line Range** | {sample.first_line} → {sample.last_line} |
| **Trace Steps** | {sample.trace_step_count} |
| **Has Code Override** | {sample.has_code_override} |
"""
    ipy_display.display(ipy_display.Markdown(metadata_md))
    # tags displayed as a bulleted list
    if sample.comma_separated_tags:
        tags = [tag.strip() for tag in sample.comma_separated_tags.split(",") if tag.strip()]
        if tags:
            tags_list = "\n".join(f"- `{tag}`" for tag in tags)
            ipy_display.display(ipy_display.Markdown(f"**Tags:**\n{tags_list}"))
        else:
            ipy_display.display(ipy_display.Markdown("**Tags:** (none)"))
    else:
        ipy_display.display(ipy_display.Markdown("**Tags:** (none)"))
    # complexity metrics displayed line-by-line with limited precision
    if sample.complexity_metrics:
        metrics_lines = []
        for key, val in sample.complexity_metrics.items():
            if isinstance(val, float):
                metrics_lines.append(f"- **{key}:** {val:.3f}")
            else:
                metrics_lines.append(f"- **{key}:** {val}")
        metrics_md = "**Complexity Metrics:**\n" + "\n".join(metrics_lines)
        ipy_display.display(ipy_display.Markdown(metrics_md))
    # description (handle multi-line by adding blockquote prefix to each line)
    ipy_display.display(ipy_display.Markdown("## Description"))
    if sample.description:
        desc_lines = sample.description.splitlines()
        desc_blockquote = "\n".join(f"> {line}" for line in desc_lines)
        ipy_display.display(ipy_display.Markdown(desc_blockquote))
    else:
        ipy_display.display(ipy_display.Markdown("> (no description)"))
    # code (with syntax highlighting)
    ipy_display.display(ipy_display.Markdown("## Code"))
    code_display = _format_long_string(sample.code, max_string_length)
    ipy_display.display(ipy_display.Markdown(f"```python\n{code_display}\n```"))
    # inputs (try dict formatting for frame_variables predict type)
    ipy_display.display(ipy_display.Markdown("## Inputs"))
    if sample.predict_type == "frame_variables":
        inputs_display, is_dict = _format_dict_string(sample.inputs, max_string_length)
        lang = "json" if is_dict else ""
    else:
        inputs_display = _format_long_string(sample.inputs, max_string_length)
        lang = ""
    ipy_display.display(ipy_display.Markdown(f"```{lang}\n{inputs_display}\n```"))
    # expected output (try dict formatting for frame_variables predict type)
    ipy_display.display(ipy_display.Markdown("## Expected Output"))
    if sample.predict_type == "frame_variables":
        output_display, is_dict = _format_dict_string(sample.expected_output, max_string_length)
        lang = "json" if is_dict else ""
    else:
        output_display = _format_long_string(sample.expected_output, max_string_length)
        lang = ""
    ipy_display.display(ipy_display.Markdown(f"```{lang}\n{output_display}\n```"))


# fetch and display the sample
if not builder_map:
    print("no builders available; run the data collection cells first")
elif viewer_target_subset not in builder_map:
    print(f"subset {viewer_target_subset!r} not found; available: {list(builder_map.keys())}")
else:
    builder = builder_map[viewer_target_subset]
    if viewer_sample_index is None:
        while True:
            picked_sample_idx = random.randint(0, len(builder) - 1)
            print(f"randomly selected sample index: {picked_sample_idx}")
            sample = builder[picked_sample_idx]
            if sample_until_code_type is not None and sample.code_type != sample_until_code_type:
                continue
            if sample_until_pred_type is not None and sample.predict_type != sample_until_pred_type:
                continue
            break
    else:
        picked_sample_idx = viewer_sample_index
        if not (0 <= picked_sample_idx < len(builder)):
            raise IndexError(f"sample index {picked_sample_idx} out of range [0, {len(builder)})")
        sample = builder[picked_sample_idx]
    ipy_display.display(ipy_display.Markdown(f"# Sample Viewer: [{viewer_target_subset}] index={picked_sample_idx}"))
    display_sample(sample, max_string_length=viewer_max_string_length)